TODO
- [ ] Handling missing record in LLM data (id 3991)

# Prepare data for EDA

Import libraries and set up paths for data loading

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
ROOT = Path.cwd().parent
ROOT

WindowsPath('C:/Users/dduya/Work/project/ev_car')

In [3]:
DATA = ROOT / "data"
INTERIM = DATA / "interim"

DATA, INTERIM

(WindowsPath('C:/Users/dduya/Work/project/ev_car/data'),
 WindowsPath('C:/Users/dduya/Work/project/ev_car/data/interim'))

In [4]:
ORIGINAL_DATA = INTERIM / "ev_cleaned_rule_based.csv"
LLM_DATA = INTERIM / "ev_extracted_gpt5nano.csv"

ORIGINAL_DATA, LLM_DATA

(WindowsPath('C:/Users/dduya/Work/project/ev_car/data/interim/ev_cleaned_rule_based.csv'),
 WindowsPath('C:/Users/dduya/Work/project/ev_car/data/interim/ev_extracted_gpt5nano.csv'))

In [5]:
df_orig = pd.read_csv(ORIGINAL_DATA)
df_llm = pd.read_csv(LLM_DATA)

In [6]:
df_orig.describe()

,Năm sản xuất,Giá_VND
count,3259.000000,3.990000e+03
mean,2024.379257,2.789441e+09
std,1.113734,3.726188e+09
min,2010.000000,9.680000e+06
25%,2024.000000,6.102500e+08
50%,2025.000000,9.100000e+08
75%,2025.000000,4.390000e+09
max,2026.000000,7.290000e+10


In [7]:
df_orig.describe

<bound method NDFrame.describe of        Ngày đăng                                          Tên xe  \
0     2026-02-28                    Xe VinFast VF8 Plus AWD 2023   
1     2026-02-28                         Xe VinFast VF8 Eco 2026   
2     2026-02-28                    Xe VinFast VF8 Plus AWD 2023   
3     2026-02-28                     Xe VinFast VF8 Eco AWD 2023   
4     2026-02-28                        Xe VinFast VF5 Plus 2024   
...          ...                                             ...   
3987  2026-01-03    VinFast VF3 2025 Xám 2000km bs hcm chính chủ   
3988  2026-01-03  VinFast Limo Green Xe Sẵn Giao Ngay Giá Ưu Đãi   
3989  2026-01-03   VINFAST VF5 TRẢ TRƯỚC 10TR – HỖ TRỢ KH NỢ XẤU   
3990  2026-01-03  VINFAST VF3 TRẢ TRƯỚC 0Đ NHẬN XE, TẶNG FULL PK   
3991  2026-02-01      VinFast Herio Green - trả trước 0Đ nhận xe   

           Tên người bán                                            Địa chỉ  \
0     GF Hoàng Quốc Việt                14 Hoàng Quốc Việt, Cầu Giấy H

In [8]:
df_llm.describe()

,id,imputed_year,imputed_mileage_km
count,3991.000000,3127.000000,1.712000e+03
mean,1995.000000,2024.166294,3.067187e+04
std,1152.246791,1.149081,3.151487e+05
min,0.000000,2010.000000,0.000000e+00
25%,997.500000,2023.000000,3.300000e+03
50%,1995.000000,2024.000000,1.400000e+04
75%,2992.500000,2025.000000,3.600000e+04
max,3990.000000,2026.000000,1.300000e+07


In [9]:
df_llm.describe

<bound method NDFrame.describe of         id                                          reasoning    brand  \
0        0  From Name: 'Xe VinFast VF8 Plus AWD 2023' impl...  VinFast   
1        1  From Name: 'Xe VinFast VF8 Eco 2026' implies b...  VinFast   
2        2  From Name: 'Xe VinFast VF8 Plus AWD 2023' impl...  VinFast   
3        3  From Name: 'Xe VinFast VF8 Eco AWD 2023' impli...  VinFast   
4        4  From Name: 'Xe VinFast VF5 Plus 2024' implies ...  VinFast   
...    ...                                                ...      ...   
3986  3985          Year 2022 in title; no odometer provided.  VinFast   
3987  3986  Odo 80 km; 'Kèm Pin- Zin 100%' phrase but not ...  VinFast   
3988  3987  Odo 2000 km; 'chính chủ' indicates used; 'Thuê...  VinFast   
3989  3988  Model appears as 'Limo Green' special variant;...  VinFast   
3990  3989  VF5 Plus appears in text; no year/mileage stated.  VinFast   

         car_model  imputed_year  imputed_mileage_km imputed_condition  \
0  

# 1. Checking for missing records

There's a mismatch in the number of records (original: 3992 records, llm: 3991 records). We need to identify the missing record in the LLM data.

Create id column in original data and sort the LLM data to identify the missing record

In [10]:
df_orig['id'] = df_orig.index

Sort the LLM data by 'id' to fix the asynchronous jumble and identify the exact missing record

In [11]:
df_llm = df_llm.sort_values(by='id').reset_index(drop=True)

In [12]:
original_ids = set(df_orig['id'])
llm_ids = set(df_llm['id'].dropna().astype(int))
missing_ids = list(original_ids - llm_ids)

missing_ids

[3991]

So the missing record in the LLM data corresponds to the original record with `id` **3991**.

In [13]:
if missing_ids:
    display(df_orig.loc[df_orig['id'] == missing_ids[0], ['id', 'Tên xe', 'Mô tả']])

,id,Tên xe,Mô tả
3991,3991,VinFast Herio Green - trả trước 0Đ nhận xe,""" LUÔN CÓ SẴN XE VÀ HỒ SƠ GIAO NGAY ""\r\n\r\nH..."


Let's just keep a note of this missing record for now and handle it later.

For this preparation for EDA, we won't need the "reasoning" column in the LLM data, so we can drop it to simplify our analysis.

In [14]:
df_llm = df_llm.drop(columns=['reasoning'])

In [15]:
df_orig.shape

(3992, 19)

In [16]:
df_llm.shape

(3991, 9)

Section 1 summary:
- We identified that the LLM data is missing one record (id 3991) compared

# 2. Data alignment

Before we can merge the two datasets, we need to ensure that they are properly aligned. We will check the data types and formats of the columns in both datasets to ensure they are compatible for merging.

In [17]:
df_orig.head()

,Ngày đăng,Tên xe,Tên người bán,Địa chỉ,Năm sản xuất,Tình trạng,Số Km đã đi,Xuất xứ,Kiểu dáng,Màu ngoại thất,Màu nội thất,Số chỗ ngồi,Số cửa,Dẫn động,Mô tả,Link,Website,Giá_VND,id
0,2026-02-28,Xe VinFast VF8 Plus AWD 2023,GF Hoàng Quốc Việt,"14 Hoàng Quốc Việt, Cầu Giấy Hà Nội",2023.0,Xe đã dùng,"50,000 Km",Lắp ráp trong nước,SUV,Đen,Đen,5 chỗ,5 cửa,AWD - 4 bánh toàn thời gian,VF8 2022 MÀU ĐEN – XE ĐẸP BẢN PLUS – QUÀ NOEL ...,https://bonbanh.com/xe-vinfast-vf8-plus-awd-20...,bonbanh.com,659000000.0,0
1,2026-02-28,Xe VinFast VF8 Eco 2026,Vinfast Times City,"Tầng B1, TTTM Vincom Times City - 458 Minh Kha...",2026.0,Xe mới,NaN,Lắp ráp trong nước,SUV,Trắng,Đen,5 chỗ,5 cửa,FWD - Dẫn động cầu trước,Vinfast VF8 (VF E35) 2026 | Xe điện mới 100% |...,https://bonbanh.com/xe-vinfast-vf8-eco-2026-65...,bonbanh.com,790000000.0,1
2,2026-02-28,Xe VinFast VF8 Plus AWD 2023,GF Hoàng Quốc Việt,"14 Hoàng Quốc Việt, Cầu Giấy Hà Nội",2023.0,Xe đã dùng,"99,000 Km",Lắp ráp trong nước,SUV,Xanh,Nhiều màu,5 chỗ,5 cửa,AWD - 4 bánh toàn thời gian,VF8 2023 MÀU Xanh– XE ĐẸP BẢN PLUS – QUÀ NOEL ...,https://bonbanh.com/xe-vinfast-vf8-plus-awd-20...,bonbanh.com,599000000.0,2
3,2026-02-28,Xe VinFast VF8 Eco AWD 2023,Đức Lộc,"Xuân Phương, Nam Từ Liêm Hà Nội",2023.0,Xe đã dùng,"55,000 Km",Lắp ráp trong nước,SUV,Xám,Đen,5 chỗ,5 cửa,AWD - 4 bánh toàn thời gian,• Giảm 70 triệu khi đổi xăng sang điện | • Tặn...,https://bonbanh.com/xe-vinfast-vf8-eco-awd-202...,bonbanh.com,580000000.0,3
4,2026-02-28,Xe VinFast VF5 Plus 2024,Tấn Nhất Tín,Hà Nội Hà Nội Xem gian hàng ⇨,2024.0,Xe đã dùng,"45,000 Km",Lắp ráp trong nước,SUV,Xám,Đen,5 chỗ,5 cửa,FWD - Dẫn động cầu trước,"Hàng mới về | Vf5 Plus mua pin 2024 odo 4,5 vạ...",https://bonbanh.com/xe-vinfast-vf5-plus-2024-6...,bonbanh.com,399000000.0,4


In [18]:
df_orig.dtypes

Ngày đăng             str
Tên xe                str
Tên người bán         str
Địa chỉ               str
Năm sản xuất      float64
Tình trạng            str
Số Km đã đi           str
Xuất xứ               str
Kiểu dáng             str
Màu ngoại thất        str
Màu nội thất          str
Số chỗ ngồi           str
Số cửa                str
Dẫn động              str
Mô tả                 str
Link                  str
Website               str
Giá_VND           float64
id                  int64
dtype: object

In [19]:
df_llm.head()

,id,brand,car_model,imputed_year,imputed_mileage_km,imputed_condition,battery_status,is_accident_free,has_aftermarket_mods
0,0,VinFast,VF8 Plus AWD,2023.0,50000.0,NaN,NaN,NaN,False
1,1,VinFast,VF8 Eco,2026.0,0.0,Mới 100%,NaN,NaN,False
2,2,VinFast,VF8 Plus AWD,2023.0,99000.0,NaN,NaN,NaN,False
3,3,VinFast,VF8 Eco AWD,2023.0,NaN,NaN,NaN,NaN,False
4,4,VinFast,VF5 Plus,2024.0,NaN,NaN,NaN,NaN,False


In [20]:
df_llm.dtypes

id                        int64
brand                       str
car_model                   str
imputed_year            float64
imputed_mileage_km      float64
imputed_condition           str
battery_status              str
is_accident_free         object
has_aftermarket_mods       bool
dtype: object

There's some datatype issue should be handled before we can merge the two datasets. The `Số Km đã đi` column in the original data is as string mixed with "Km". We need to clean this column to convert it into a numeric format for proper analysis.

In [21]:
import numpy as np

In [22]:
df_orig['orig_mileage'] = (df_orig['Số Km đã đi']
                           .astype(str)
                           .str.replace(r'[kK]m', '', regex=True)
                           .str.replace(',', '', regex=True)
                           .str.strip()
                           )

In [23]:
df_orig['orig_mileage'] = pd.to_numeric(df_orig['orig_mileage'], errors='coerce')

In [24]:
df_orig['orig_mileage'].describe()

count    1.773000e+03
mean     3.536720e+04
std      1.803710e+05
min      0.000000e+00
25%      5.100000e+03
50%      2.000000e+04
75%      4.330000e+04
max      7.300000e+06
Name: orig_mileage, dtype: float64

In [25]:
df_orig.dtypes

Ngày đăng             str
Tên xe                str
Tên người bán         str
Địa chỉ               str
Năm sản xuất      float64
Tình trạng            str
Số Km đã đi           str
Xuất xứ               str
Kiểu dáng             str
Màu ngoại thất        str
Màu nội thất          str
Số chỗ ngồi           str
Số cửa                str
Dẫn động              str
Mô tả                 str
Link                  str
Website               str
Giá_VND           float64
id                  int64
orig_mileage      float64
dtype: object

In [26]:
df_orig.head()

,Ngày đăng,Tên xe,Tên người bán,Địa chỉ,Năm sản xuất,Tình trạng,Số Km đã đi,Xuất xứ,Kiểu dáng,Màu ngoại thất,Màu nội thất,Số chỗ ngồi,Số cửa,Dẫn động,Mô tả,Link,Website,Giá_VND,id,orig_mileage
0,2026-02-28,Xe VinFast VF8 Plus AWD 2023,GF Hoàng Quốc Việt,"14 Hoàng Quốc Việt, Cầu Giấy Hà Nội",2023.0,Xe đã dùng,"50,000 Km",Lắp ráp trong nước,SUV,Đen,Đen,5 chỗ,5 cửa,AWD - 4 bánh toàn thời gian,VF8 2022 MÀU ĐEN – XE ĐẸP BẢN PLUS – QUÀ NOEL ...,https://bonbanh.com/xe-vinfast-vf8-plus-awd-20...,bonbanh.com,659000000.0,0,50000.0
1,2026-02-28,Xe VinFast VF8 Eco 2026,Vinfast Times City,"Tầng B1, TTTM Vincom Times City - 458 Minh Kha...",2026.0,Xe mới,NaN,Lắp ráp trong nước,SUV,Trắng,Đen,5 chỗ,5 cửa,FWD - Dẫn động cầu trước,Vinfast VF8 (VF E35) 2026 | Xe điện mới 100% |...,https://bonbanh.com/xe-vinfast-vf8-eco-2026-65...,bonbanh.com,790000000.0,1,NaN
2,2026-02-28,Xe VinFast VF8 Plus AWD 2023,GF Hoàng Quốc Việt,"14 Hoàng Quốc Việt, Cầu Giấy Hà Nội",2023.0,Xe đã dùng,"99,000 Km",Lắp ráp trong nước,SUV,Xanh,Nhiều màu,5 chỗ,5 cửa,AWD - 4 bánh toàn thời gian,VF8 2023 MÀU Xanh– XE ĐẸP BẢN PLUS – QUÀ NOEL ...,https://bonbanh.com/xe-vinfast-vf8-plus-awd-20...,bonbanh.com,599000000.0,2,99000.0
3,2026-02-28,Xe VinFast VF8 Eco AWD 2023,Đức Lộc,"Xuân Phương, Nam Từ Liêm Hà Nội",2023.0,Xe đã dùng,"55,000 Km",Lắp ráp trong nước,SUV,Xám,Đen,5 chỗ,5 cửa,AWD - 4 bánh toàn thời gian,• Giảm 70 triệu khi đổi xăng sang điện | • Tặn...,https://bonbanh.com/xe-vinfast-vf8-eco-awd-202...,bonbanh.com,580000000.0,3,55000.0
4,2026-02-28,Xe VinFast VF5 Plus 2024,Tấn Nhất Tín,Hà Nội Hà Nội Xem gian hàng ⇨,2024.0,Xe đã dùng,"45,000 Km",Lắp ráp trong nước,SUV,Xám,Đen,5 chỗ,5 cửa,FWD - Dẫn động cầu trước,"Hàng mới về | Vf5 Plus mua pin 2024 odo 4,5 vạ...",https://bonbanh.com/xe-vinfast-vf5-plus-2024-6...,bonbanh.com,399000000.0,4,45000.0


Next is the `Tình trạng` - car condition column. We need to check the consistency of the categories in this column and ensure that they are properly aligned between the two datasets for accurate merging.

In [27]:
raw_cond = df_orig['Tình trạng'].unique()

raw_cond

<StringArray>
[    'Xe đã dùng',         'Xe mới', 'Đã qua sử dụng',       'Mới 100%',
        'Mới 99%',              nan,            'Mới',     'Đã sử dụng']
Length: 8, dtype: str

In [28]:
cond_map = {
    'Xe đã dùng': 'Used',
    'Xe mới': 'New',
    'Đã qua sử dụng': 'Used',
    'Mới 99%': 'Used',
    'Mới 100%': 'New',
    'Mới': 'New',
    'Đã sử dụng': 'Used'
}

In [29]:
df_orig['orig_condition'] = df_orig['Tình trạng'].map(cond_map)

In [30]:
df_orig['orig_condition'].unique()

<StringArray>
['Used', 'New', nan]
Length: 3, dtype: str

In [31]:
print(df_orig.head())

    Ngày đăng                        Tên xe       Tên người bán  \
0  2026-02-28  Xe VinFast VF8 Plus AWD 2023  GF Hoàng Quốc Việt   
1  2026-02-28       Xe VinFast VF8 Eco 2026  Vinfast Times City   
2  2026-02-28  Xe VinFast VF8 Plus AWD 2023  GF Hoàng Quốc Việt   
3  2026-02-28   Xe VinFast VF8 Eco AWD 2023             Đức Lộc   
4  2026-02-28      Xe VinFast VF5 Plus 2024        Tấn Nhất Tín   

                                             Địa chỉ  Năm sản xuất  \
0                14 Hoàng Quốc Việt, Cầu Giấy Hà Nội        2023.0   
1  Tầng B1, TTTM Vincom Times City - 458 Minh Kha...        2026.0   
2                14 Hoàng Quốc Việt, Cầu Giấy Hà Nội        2023.0   
3                    Xuân Phương, Nam Từ Liêm Hà Nội        2023.0   
4                     Hà Nội Hà Nội Xem gian hàng  ⇨        2024.0   

   Tình trạng Số Km đã đi             Xuất xứ Kiểu dáng Màu ngoại thất  ...  \
0  Xe đã dùng   50,000 Km  Lắp ráp trong nước       SUV            Đen  ...   
1      Xe mới     

Let's do the same to the LLM data to ensure that the `condition` column is also properly aligned with the original data for accurate merging.

In [32]:
df_llm['imputed_condition'].unique()

<StringArray>
[nan, 'Mới 100%', 'Đã qua sử dụng']
Length: 3, dtype: str

In [33]:
llm_cond_map = {
    'Mới 100%': 'New',
    'Đã qua sử dụng': 'Used'
}

In [34]:
df_llm['imputed_condition'] = df_llm['imputed_condition'].map(llm_cond_map)

In [35]:
df_llm['imputed_condition'].unique()

<StringArray>
[nan, 'New', 'Used']
Length: 3, dtype: str

# 3. Merge and fill missing values

In [36]:
df_merged = pd.merge(df_orig, df_llm, on='id', how='left')

In [37]:
df_merged.shape

(3992, 29)

In [38]:
df_merged.head()

,Ngày đăng,Tên xe,Tên người bán,Địa chỉ,Năm sản xuất,Tình trạng,Số Km đã đi,Xuất xứ,Kiểu dáng,Màu ngoại thất,...,orig_mileage,orig_condition,brand,car_model,imputed_year,imputed_mileage_km,imputed_condition,battery_status,is_accident_free,has_aftermarket_mods
0,2026-02-28,Xe VinFast VF8 Plus AWD 2023,GF Hoàng Quốc Việt,"14 Hoàng Quốc Việt, Cầu Giấy Hà Nội",2023.0,Xe đã dùng,"50,000 Km",Lắp ráp trong nước,SUV,Đen,...,50000.0,Used,VinFast,VF8 Plus AWD,2023.0,50000.0,NaN,NaN,NaN,False
1,2026-02-28,Xe VinFast VF8 Eco 2026,Vinfast Times City,"Tầng B1, TTTM Vincom Times City - 458 Minh Kha...",2026.0,Xe mới,NaN,Lắp ráp trong nước,SUV,Trắng,...,NaN,New,VinFast,VF8 Eco,2026.0,0.0,New,NaN,NaN,False
2,2026-02-28,Xe VinFast VF8 Plus AWD 2023,GF Hoàng Quốc Việt,"14 Hoàng Quốc Việt, Cầu Giấy Hà Nội",2023.0,Xe đã dùng,"99,000 Km",Lắp ráp trong nước,SUV,Xanh,...,99000.0,Used,VinFast,VF8 Plus AWD,2023.0,99000.0,NaN,NaN,NaN,False
3,2026-02-28,Xe VinFast VF8 Eco AWD 2023,Đức Lộc,"Xuân Phương, Nam Từ Liêm Hà Nội",2023.0,Xe đã dùng,"55,000 Km",Lắp ráp trong nước,SUV,Xám,...,55000.0,Used,VinFast,VF8 Eco AWD,2023.0,NaN,NaN,NaN,NaN,False
4,2026-02-28,Xe VinFast VF5 Plus 2024,Tấn Nhất Tín,Hà Nội Hà Nội Xem gian hàng ⇨,2024.0,Xe đã dùng,"45,000 Km",Lắp ráp trong nước,SUV,Xám,...,45000.0,Used,VinFast,VF5 Plus,2024.0,NaN,NaN,NaN,NaN,False


In [39]:
df_merged.loc[df_merged['id'] == 3991, ['id', 'brand', 'imputed_year']]

,id,brand,imputed_year
3991,3991,NaN,NaN


In [40]:
df_merged['Năm sản xuất'] = pd.to_numeric(df_merged['Năm sản xuất'], errors='coerce')

In [41]:
df_merged['final_year'] = df_merged['Năm sản xuất'].fillna(df_merged['imputed_year'])

In [42]:
df_merged['final_year'].describe()

count    3923.000000
mean     2024.297731
std         1.116636
min      2010.000000
25%      2024.000000
50%      2025.000000
75%      2025.000000
max      2026.000000
Name: final_year, dtype: float64

In [43]:
df_merged['final_mileage'] = df_merged['orig_mileage'].fillna(df_merged['imputed_mileage_km'])

In [44]:
df_merged['final_condition'] = df_merged['orig_condition'].fillna(df_merged['imputed_condition'])

In [45]:
df_merged['final_brand'] = df_merged['brand']
df_merged['final_car_model'] = df_merged['car_model']

In [46]:
print(
    f"Original missing years: {df_orig['Năm sản xuất'].isna().sum()} -> Final missing: {df_merged['final_year'].isna().sum()}")

Original missing years: 733 -> Final missing: 69


In [47]:
print(f"Original missing mileage: {df_orig['orig_mileage'].isna().sum()} "
      f"-> Final missing: {df_merged['final_mileage'].isna().sum()}")
print(f"Original missing condition: {df_orig['orig_condition'].isna().sum()} "
      f"-> Final missing: {df_merged['final_condition'].isna().sum()}")
print(f"Original missing condition: {df_orig['orig_condition'].isna().sum()} "
      f"-> Final missing: {df_merged['final_condition'].isna().sum()}")

Original missing mileage: 2219 -> Final missing: 1556
Original missing condition: 728 -> Final missing: 465
Original missing condition: 728 -> Final missing: 465


In [48]:
df_merged.head()

,Ngày đăng,Tên xe,Tên người bán,Địa chỉ,Năm sản xuất,Tình trạng,Số Km đã đi,Xuất xứ,Kiểu dáng,Màu ngoại thất,...,imputed_mileage_km,imputed_condition,battery_status,is_accident_free,has_aftermarket_mods,final_year,final_mileage,final_condition,final_brand,final_car_model
0,2026-02-28,Xe VinFast VF8 Plus AWD 2023,GF Hoàng Quốc Việt,"14 Hoàng Quốc Việt, Cầu Giấy Hà Nội",2023.0,Xe đã dùng,"50,000 Km",Lắp ráp trong nước,SUV,Đen,...,50000.0,NaN,NaN,NaN,False,2023.0,50000.0,Used,VinFast,VF8 Plus AWD
1,2026-02-28,Xe VinFast VF8 Eco 2026,Vinfast Times City,"Tầng B1, TTTM Vincom Times City - 458 Minh Kha...",2026.0,Xe mới,NaN,Lắp ráp trong nước,SUV,Trắng,...,0.0,New,NaN,NaN,False,2026.0,0.0,New,VinFast,VF8 Eco
2,2026-02-28,Xe VinFast VF8 Plus AWD 2023,GF Hoàng Quốc Việt,"14 Hoàng Quốc Việt, Cầu Giấy Hà Nội",2023.0,Xe đã dùng,"99,000 Km",Lắp ráp trong nước,SUV,Xanh,...,99000.0,NaN,NaN,NaN,False,2023.0,99000.0,Used,VinFast,VF8 Plus AWD
3,2026-02-28,Xe VinFast VF8 Eco AWD 2023,Đức Lộc,"Xuân Phương, Nam Từ Liêm Hà Nội",2023.0,Xe đã dùng,"55,000 Km",Lắp ráp trong nước,SUV,Xám,...,NaN,NaN,NaN,NaN,False,2023.0,55000.0,Used,VinFast,VF8 Eco AWD
4,2026-02-28,Xe VinFast VF5 Plus 2024,Tấn Nhất Tín,Hà Nội Hà Nội Xem gian hàng ⇨,2024.0,Xe đã dùng,"45,000 Km",Lắp ráp trong nước,SUV,Xám,...,NaN,NaN,NaN,NaN,False,2024.0,45000.0,Used,VinFast,VF5 Plus


Summary:

# 4. Clean up

In [49]:
cols_to_drop = ['Tên xe',
                'Năm sản xuất',
                'Số Km đã đi',
                'Tình trạng',
                'orig_mileage',
                'orig_condition',
                'brand',
                'car_model',
                'imputed_year',
                'imputed_mileage_km',
                'imputed_condition',
                'Link',
                'Mô tả'
                ]

In [50]:
df_final = df_merged.drop(columns=cols_to_drop, errors='ignore')

In [51]:
rename_dict = {
    'Ngày đăng': 'post_date',
    'Tên người bán': 'seller_name',
    'Địa chỉ': 'location',
    'Xuất xứ': 'origin',
    'Kiểu dáng': 'body_type',
    'Màu ngoại thất': 'exterior_color',
    'Màu nội thất': 'interior_color',
    'Số chỗ ngồi': 'seats',
    'Số cửa': 'doors',
    'Dẫn động': 'drivetrain',
    'Website': 'website',
    'Giá_VND': 'price_vnd',

    'final_year': 'year',
    'final_mileage': 'mileage_km',
    'final_condition': 'condition',
    'final_brand': 'brand',
    'final_car_model': 'car_model'
}

In [52]:
df_final = df_final.rename(columns=rename_dict)

In [53]:
df_final.head()

,post_date,seller_name,location,origin,body_type,exterior_color,interior_color,seats,doors,drivetrain,...,price_vnd,id,battery_status,is_accident_free,has_aftermarket_mods,year,mileage_km,condition,brand,car_model
0,2026-02-28,GF Hoàng Quốc Việt,"14 Hoàng Quốc Việt, Cầu Giấy Hà Nội",Lắp ráp trong nước,SUV,Đen,Đen,5 chỗ,5 cửa,AWD - 4 bánh toàn thời gian,...,659000000.0,0,NaN,NaN,False,2023.0,50000.0,Used,VinFast,VF8 Plus AWD
1,2026-02-28,Vinfast Times City,"Tầng B1, TTTM Vincom Times City - 458 Minh Kha...",Lắp ráp trong nước,SUV,Trắng,Đen,5 chỗ,5 cửa,FWD - Dẫn động cầu trước,...,790000000.0,1,NaN,NaN,False,2026.0,0.0,New,VinFast,VF8 Eco
2,2026-02-28,GF Hoàng Quốc Việt,"14 Hoàng Quốc Việt, Cầu Giấy Hà Nội",Lắp ráp trong nước,SUV,Xanh,Nhiều màu,5 chỗ,5 cửa,AWD - 4 bánh toàn thời gian,...,599000000.0,2,NaN,NaN,False,2023.0,99000.0,Used,VinFast,VF8 Plus AWD
3,2026-02-28,Đức Lộc,"Xuân Phương, Nam Từ Liêm Hà Nội",Lắp ráp trong nước,SUV,Xám,Đen,5 chỗ,5 cửa,AWD - 4 bánh toàn thời gian,...,580000000.0,3,NaN,NaN,False,2023.0,55000.0,Used,VinFast,VF8 Eco AWD
4,2026-02-28,Tấn Nhất Tín,Hà Nội Hà Nội Xem gian hàng ⇨,Lắp ráp trong nước,SUV,Xám,Đen,5 chỗ,5 cửa,FWD - Dẫn động cầu trước,...,399000000.0,4,NaN,NaN,False,2024.0,45000.0,Used,VinFast,VF5 Plus


In [54]:
print(df_final.head())

    post_date         seller_name  \
0  2026-02-28  GF Hoàng Quốc Việt   
1  2026-02-28  Vinfast Times City   
2  2026-02-28  GF Hoàng Quốc Việt   
3  2026-02-28             Đức Lộc   
4  2026-02-28        Tấn Nhất Tín   

                                            location              origin  \
0                14 Hoàng Quốc Việt, Cầu Giấy Hà Nội  Lắp ráp trong nước   
1  Tầng B1, TTTM Vincom Times City - 458 Minh Kha...  Lắp ráp trong nước   
2                14 Hoàng Quốc Việt, Cầu Giấy Hà Nội  Lắp ráp trong nước   
3                    Xuân Phương, Nam Từ Liêm Hà Nội  Lắp ráp trong nước   
4                     Hà Nội Hà Nội Xem gian hàng  ⇨  Lắp ráp trong nước   

  body_type exterior_color interior_color  seats  doors  \
0       SUV            Đen            Đen  5 chỗ  5 cửa   
1       SUV          Trắng            Đen  5 chỗ  5 cửa   
2       SUV           Xanh      Nhiều màu  5 chỗ  5 cửa   
3       SUV            Xám            Đen  5 chỗ  5 cửa   
4       SUV            Xá

In [55]:
df_final['seats'] = df_final['seats'].astype(str).str.extract(r'(\d+)').astype(float)
df_final['doors'] = df_final['doors'].astype(str).str.extract(r'(\d+)').astype(float)

In [56]:
df_final['brand'] = df_final['brand'].astype(str).str.title()

In [57]:
df_final['brand'].unique()

<StringArray>
[      'Vinfast', 'Mercedes-Benz',       'Bestune',          'Ford',
 'Mercedes Benz',        'Hongqi',       'Porsche',         'Geely',
         'Volvo',           'Byd',         'Haima',          'Mini',
          'Audi',           'Bmw',        'Wuling',      'Dongfeng',
         'Minio',       'Hyundai',        'Jaguar',             nan,
            'Ec',          'Limo',         'Herio',      'Mercedes',
         'Nerio',        'Toyota',    'Limo Green',         '/Null',
        'Jaecoo',         'Lexus',        'Vinfat']
Length: 31, dtype: str

In [58]:
acronyms = {
    'Bmw': 'BMW',
    'Byd': 'BYD',
    'Mg': 'MG',
    'Mini': 'MINI',
    'Nan': np.nan
}

In [59]:
df_final['brand'] = df_final['brand'].replace(acronyms)

In [60]:
df_final['brand'].unique()

<StringArray>
[      'Vinfast', 'Mercedes-Benz',       'Bestune',          'Ford',
 'Mercedes Benz',        'Hongqi',       'Porsche',         'Geely',
         'Volvo',           'BYD',         'Haima',          'MINI',
          'Audi',           'BMW',        'Wuling',      'Dongfeng',
         'Minio',       'Hyundai',        'Jaguar',             nan,
            'Ec',          'Limo',         'Herio',      'Mercedes',
         'Nerio',        'Toyota',    'Limo Green',         '/Null',
        'Jaecoo',         'Lexus',        'Vinfat']
Length: 31, dtype: str

In [61]:
junk_values = ['Nan', '/Null', 'None']
df_final['brand'] = df_final['brand'].replace(junk_values, np.nan)

In [62]:
typo_mapping = {
    'Vinfat': 'VinFast',
    'Vinfast': 'VinFast',
    'Mercedes Benz': 'Mercedes-Benz',
    'Mercedes': 'Mercedes-Benz',
}

In [63]:
df_final['brand'] = df_final['brand'].replace(typo_mapping)

In [64]:
df_final['brand'].unique()

<StringArray>
[      'VinFast', 'Mercedes-Benz',       'Bestune',          'Ford',
        'Hongqi',       'Porsche',         'Geely',         'Volvo',
           'BYD',         'Haima',          'MINI',          'Audi',
           'BMW',        'Wuling',      'Dongfeng',         'Minio',
       'Hyundai',        'Jaguar',             nan,            'Ec',
          'Limo',         'Herio',         'Nerio',        'Toyota',
    'Limo Green',        'Jaecoo',         'Lexus']
Length: 27, dtype: str

In [65]:
leaked_models = ['Limo', 'Limo Green', 'Herio', 'Nerio', 'Ec', 'Minio']

In [66]:
bleed_mask = df_final['brand'].isin(leaked_models)

In [67]:
df_final.loc[bleed_mask, 'car_model'] = df_final.loc[bleed_mask, 'brand']

In [68]:
df_final.loc[bleed_mask, 'brand'] = 'VinFast'

In [69]:
df_final["brand"].unique()

<StringArray>
[      'VinFast', 'Mercedes-Benz',       'Bestune',          'Ford',
        'Hongqi',       'Porsche',         'Geely',         'Volvo',
           'BYD',         'Haima',          'MINI',          'Audi',
           'BMW',        'Wuling',      'Dongfeng',       'Hyundai',
        'Jaguar',             nan,        'Toyota',        'Jaecoo',
         'Lexus']
Length: 21, dtype: str

In [70]:
len(df_final["location"].unique())

1107

In [71]:
province_mapping = {
    # The Big 5 centrally governed cities
    'hà nội': 'Hà Nội', 'ha noi': 'Hà Nội', 'hn': 'Hà Nội',
    'hồ chí minh': 'Hồ Chí Minh', 'hcm': 'Hồ Chí Minh', 'sài gòn': 'Hồ Chí Minh', 'tphcm': 'Hồ Chí Minh',
    'đà nẵng': 'Đà Nẵng', 'da nang': 'Đà Nẵng',
    'hải phòng': 'Hải Phòng', 'hai phong': 'Hải Phòng',
    'cần thơ': 'Cần Thơ', 'can tho': 'Cần Thơ',

    # Major Provinces
    'vĩnh phúc': 'Vĩnh Phúc',
    'phú thọ': 'Phú Thọ',
    'long an': 'Long An',
    'cao bằng': 'Cao Bằng',
    'đồng nai': 'Đồng Nai', 'dong nai': 'Đồng Nai',
    'khánh hòa': 'Khánh Hòa', 'khánh hoà': 'Khánh Hòa',
    'thanh hóa': 'Thanh Hóa', 'thanh hoá': 'Thanh Hóa',
    'ninh bình': 'Ninh Bình',
    'thái nguyên': 'Thái Nguyên',
    'bắc ninh': 'Bắc Ninh',
    'thái bình': 'Thái Bình',
    'tiền giang': 'Tiền Giang',
    'lâm đồng': 'Lâm Đồng',
    'đắk lắk': 'Đắk Lắk', 'đăk lăk': 'Đắk Lắk', 'dak lak': 'Đắk Lắk',
    'an giang': 'An Giang',
    'tây ninh': 'Tây Ninh',
    'hưng yên': 'Hưng Yên',
    'hải dương': 'Hải Dương',
    'phú yên': 'Phú Yên',
    'cà mau': 'Cà Mau',
    'thừa thiên huế': 'Thừa Thiên Huế', 'huế': 'Thừa Thiên Huế',
    'gia lai': 'Gia Lai',
    'lạng sơn': 'Lạng Sơn',
    'hà tĩnh': 'Hà Tĩnh',
    'hà nam': 'Hà Nam',
    'tuyên quang': 'Tuyên Quang',
    'điện biên': 'Điện Biên',
    'nam định': 'Nam Định',
    'hòa bình': 'Hòa Bình', 'hoà bình': 'Hòa Bình',
    'bình phước': 'Bình Phước',
    'bến tre': 'Bến Tre',
    'kiên giang': 'Kiên Giang',
    'quảng trị': 'Quảng Trị',
    'bắc giang': 'Bắc Giang',
    'quảng nam': 'Quảng Nam',
    'bình thuận': 'Bình Thuận',
    'bình định': 'Bình Định',
    'quảng bình': 'Quảng Bình',
    'sóc trăng': 'Sóc Trăng',
    'đồng tháp': 'Đồng Tháp',
    'vĩnh long': 'Vĩnh Long',
    'hà giang': 'Hà Giang'
}

In [72]:
def extract_province(address):
    if pd.isna(address):
        return np.nan

    addr_lower = str(address).lower()

    # Priority Scan: Check if any official province name is inside the string
    for key, official_name in province_mapping.items():
        if key in addr_lower:
            return official_name

    # Edge Cases:
    # "Quận 8" -> HCM
    if 'quận 8' in addr_lower:
        return 'Hồ Chí Minh'

    return 'Tỉnh/Thành Khác'

In [73]:
df_final['city'] = df_final['location'].apply(extract_province)

In [74]:
df_final = df_final.drop(columns=['location'])

In [75]:
df_final['city'].unique()

<StringArray>
[         'Hà Nội',         'Cần Thơ',     'Hồ Chí Minh', 'Tỉnh/Thành Khác',
       'Khánh Hòa',        'Hòa Bình',       'Thanh Hóa',         'Long An',
       'Vĩnh Phúc',         'Phú Thọ',        'Bắc Ninh',        'Lâm Đồng',
       'Ninh Bình',       'Thái Bình',        'Lạng Sơn',       'Hải Phòng',
        'Hưng Yên',         'Hà Tĩnh',     'Thái Nguyên',          'Hà Nam',
         'Đà Nẵng',     'Tuyên Quang',       'Điện Biên',        'An Giang',
         'Đắk Lắk',      'Bình Phước',         'Bến Tre',        'Đồng Nai',
       'Hải Dương',         'Gia Lai',      'Kiên Giang',         'Phú Yên',
        'Cao Bằng',       'Bắc Giang',       'Quảng Trị',  'Thừa Thiên Huế',
      'Tiền Giang',          'Cà Mau',        'Tây Ninh',        'Nam Định',
       'Quảng Nam',      'Bình Thuận',       'Bình Định',      'Quảng Bình',
       'Sóc Trăng',       'Đồng Tháp',       'Vĩnh Long',        'Hà Giang']
Length: 48, dtype: str

In [76]:
df_final

,post_date,seller_name,origin,body_type,exterior_color,interior_color,seats,doors,drivetrain,website,...,id,battery_status,is_accident_free,has_aftermarket_mods,year,mileage_km,condition,brand,car_model,city
0,2026-02-28,GF Hoàng Quốc Việt,Lắp ráp trong nước,SUV,Đen,Đen,5.0,5.0,AWD - 4 bánh toàn thời gian,bonbanh.com,...,0,NaN,NaN,False,2023.0,50000.0,Used,VinFast,VF8 Plus AWD,Hà Nội
1,2026-02-28,Vinfast Times City,Lắp ráp trong nước,SUV,Trắng,Đen,5.0,5.0,FWD - Dẫn động cầu trước,bonbanh.com,...,1,NaN,NaN,False,2026.0,0.0,New,VinFast,VF8 Eco,Hà Nội
2,2026-02-28,GF Hoàng Quốc Việt,Lắp ráp trong nước,SUV,Xanh,Nhiều màu,5.0,5.0,AWD - 4 bánh toàn thời gian,bonbanh.com,...,2,NaN,NaN,False,2023.0,99000.0,Used,VinFast,VF8 Plus AWD,Hà Nội
3,2026-02-28,Đức Lộc,Lắp ráp trong nước,SUV,Xám,Đen,5.0,5.0,AWD - 4 bánh toàn thời gian,bonbanh.com,...,3,NaN,NaN,False,2023.0,55000.0,Used,VinFast,VF8 Eco AWD,Hà Nội
4,2026-02-28,Tấn Nhất Tín,Lắp ráp trong nước,SUV,Xám,Đen,5.0,5.0,FWD - Dẫn động cầu trước,bonbanh.com,...,4,NaN,NaN,False,2024.0,45000.0,Used,VinFast,VF5 Plus,Hà Nội
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3987,2026-01-03,Hữu Trung,Đang cập nhật,NaN,Xám,NaN,4.0,NaN,NaN,chotot.com,...,3987,NaN,NaN,False,2024.0,2000.0,Used,VinFast,VF3,Hồ Chí Minh
3988,2026-01-03,VINFAST MIỀN NAM,Việt Nam,Minivan (MPV),Màu khác,NaN,7.0,NaN,NaN,chotot.com,...,3988,NaN,NaN,False,2026.0,NaN,New,VinFast,Limo Green,Hồ Chí Minh
3989,2026-01-03,VINFAST MIỀN NAM,Việt Nam,SUV / Cross over,Màu khác,NaN,5.0,NaN,NaN,chotot.com,...,3989,NaN,NaN,False,2025.0,NaN,New,VinFast,VF5 Plus,Hồ Chí Minh
3990,2026-01-03,VINFAST MIỀN NAM,Việt Nam,NaN,Màu khác,NaN,4.0,NaN,NaN,chotot.com,...,3990,NaN,NaN,True,2026.0,210.0,New,VinFast,VF3,Hồ Chí Minh


In [77]:
df_final["car_model"].value_counts()

car_model
VF3                         470
VF5 Plus                    429
VF3 Plus                    255
VF8 Plus AWD                204
VF6 Plus                    191
                           ... 
VF6 Plus 25 (Bản NC)          1
VF9-Plus-trần thép-7 chỗ      1
MPV 7 chỗ                     1
VF MPV 7 chỗ                  1
Ec Van NC                     1
Name: count, Length: 336, dtype: int64

In [78]:
import re

In [79]:
def clean_car_model(model_str):
    if pd.isna(model_str):
        return np.nan

    m = str(model_str).upper()

    # Group 1: The VinFast VF Series (Matches VF3, VF 3, VFE34, VF e34)
    vf_match = re.search(r'VF\s*E?\s*(\d+|34)', m)
    if vf_match:
        number = vf_match.group(1)
        if number == '34': return 'VF e34'
        return f'VF{number}'

    # Group 2: The VinFast Custom Models
    if 'LIMO GREEN' in m or 'LIMOGREEN' in m: return 'Limo Green'
    if 'HERIO' in m: return 'Herio Green'
    if 'NERIO' in m: return 'Herio Green'  # Fixing the typo
    if 'MINIO' in m: return 'Minio Green'
    if 'EC VAN' in m or 'EC' == m: return 'EC Van'
    if 'MPV 7' in m or 'MPV7' in m or 'MVP7' in m: return 'VF MPV 7'

    # Group 3: BYD Models
    if 'ATTO 2' in m: return 'Atto 2'
    if 'ATTO 3' in m: return 'Atto 3'
    if 'SEALION 6' in m: return 'Sealion 6'
    if 'SEALION 8' in m or 'SEALION 08' in m: return 'Sealion 8'
    if 'SEAL' in m: return 'Seal'
    if 'DOLPHIN' in m: return 'Dolphin'
    if 'M6' in m: return 'M6'
    if 'HAN' in m: return 'Han'

    # Group 4: Mercedes EQ Series
    eq_match = re.search(r'(EQ[A-Z]\s*\d+)', m)
    if eq_match:
        # Standardize space (e.g. EQB250 -> EQB 250)
        return re.sub(r'([A-Z])(\d)', r'\1 \2', eq_match.group(1))

    # Group 5: Porsche Taycan & Macan
    if 'TAYCAN' in m: return 'Taycan'
    if 'MACAN' in m: return 'Macan'

    # Group 6: Wuling
    if 'BINGO' in m: return 'Bingo'
    if 'MINI' in m and 'HONGGUANG' in m: return 'Mini EV'
    if 'XIAOMA' in m: return 'Xiaoma'

    # Fallback: Just return the first two words capitalized if no match
    parts = m.split()
    if len(parts) >= 2:
        return f"{parts[0]} {parts[1]}".title()
    return m.title()

In [80]:
df_final['car_model_clean'] = df_final['car_model'].apply(clean_car_model)

In [81]:
df_final['car_model_clean'].value_counts()

car_model_clean
VF8        851
VF3        770
VF5        632
VF6        424
VF9        327
          ... 
J7 Awd       1
900Lx        1
Mini Ev      1
Lm500H       1
Mvp 7        1
Name: count, Length: 71, dtype: int64

In [82]:
df_final['car_model_clean'] = df_final['car_model_clean'].replace(junk_values, np.nan)

In [83]:
df_final[df_final['car_model_clean'] == "Ô Tô"]

,post_date,seller_name,origin,body_type,exterior_color,interior_color,seats,doors,drivetrain,website,...,battery_status,is_accident_free,has_aftermarket_mods,year,mileage_km,condition,brand,car_model,city,car_model_clean
3307,2025-12-22,Nhật Hào,Đang cập nhật,NaN,Hồng,NaN,NaN,NaN,NaN,chotot.com,...,NaN,NaN,True,2023.0,35000.0,Used,Wuling,Ô tô điện Hồng,Vĩnh Long,Ô Tô


In [84]:
df_orig[df_orig['Tên người bán'] == "Nhật Hào"]

,Ngày đăng,Tên xe,Tên người bán,Địa chỉ,Năm sản xuất,Tình trạng,Số Km đã đi,Xuất xứ,Kiểu dáng,Màu ngoại thất,...,Số chỗ ngồi,Số cửa,Dẫn động,Mô tả,Link,Website,Giá_VND,id,orig_mileage,orig_condition
3300,2025-12-21,VinFast Limo Green 2025,Nhật Hào,"Phường 9, Thành phố Mỹ Tho, Tiền Giang",2025.0,Mới,NaN,Việt Nam,NaN,NaN,...,7,NaN,NaN,"✨ VinFast Limo Green, mẫu xe điện hoàn toàn mớ...",https://xe.chotot.com/mua-ban-oto-thanh-pho-my...,chotot.com,7.090000e+07,3300,NaN,New
3301,2025-12-21,VinFast VF3 2025,Nhật Hào,"Phường 9, Thành phố Mỹ Tho, Tiền Giang",2025.0,Mới,NaN,Việt Nam,NaN,Màu khác,...,4,NaN,NaN,"Cuối năm xả kho, xe có sẵn đủ tất cả loại màu,...",https://xe.chotot.com/mua-ban-oto-thanh-pho-my...,chotot.com,2.750000e+09,3301,NaN,New
3307,2025-12-22,Wuling Ô tô điện Hồng,Nhật Hào,"Phường 1, Thành phố Vĩnh Long, Vĩnh Long",2023.0,Đã sử dụng,35000,Đang cập nhật,NaN,Hồng,...,NaN,NaN,NaN,"Wuling màu hồng còn như mới, nữ chạy kỹ.\r\nOd...",https://xe.chotot.com/mua-ban-oto-thanh-pho-vi...,chotot.com,2.000000e+09,3307,35000.0,Used


In [85]:
junk_models = ["Ô Tô", "/Null", "Unknown"]

In [86]:
df_final = df_final[~df_final['car_model_clean'].isin(junk_models)]

In [87]:
df_final['car_model_clean'].value_counts()

car_model_clean
VF8        851
VF3        770
VF5        632
VF6        424
VF9        327
          ... 
J7 Awd       1
900Lx        1
Mini Ev      1
Lm500H       1
Mvp 7        1
Name: count, Length: 69, dtype: int64

In [88]:
df_final.head()

,post_date,seller_name,origin,body_type,exterior_color,interior_color,seats,doors,drivetrain,website,...,battery_status,is_accident_free,has_aftermarket_mods,year,mileage_km,condition,brand,car_model,city,car_model_clean
0,2026-02-28,GF Hoàng Quốc Việt,Lắp ráp trong nước,SUV,Đen,Đen,5.0,5.0,AWD - 4 bánh toàn thời gian,bonbanh.com,...,NaN,NaN,False,2023.0,50000.0,Used,VinFast,VF8 Plus AWD,Hà Nội,VF8
1,2026-02-28,Vinfast Times City,Lắp ráp trong nước,SUV,Trắng,Đen,5.0,5.0,FWD - Dẫn động cầu trước,bonbanh.com,...,NaN,NaN,False,2026.0,0.0,New,VinFast,VF8 Eco,Hà Nội,VF8
2,2026-02-28,GF Hoàng Quốc Việt,Lắp ráp trong nước,SUV,Xanh,Nhiều màu,5.0,5.0,AWD - 4 bánh toàn thời gian,bonbanh.com,...,NaN,NaN,False,2023.0,99000.0,Used,VinFast,VF8 Plus AWD,Hà Nội,VF8
3,2026-02-28,Đức Lộc,Lắp ráp trong nước,SUV,Xám,Đen,5.0,5.0,AWD - 4 bánh toàn thời gian,bonbanh.com,...,NaN,NaN,False,2023.0,55000.0,Used,VinFast,VF8 Eco AWD,Hà Nội,VF8
4,2026-02-28,Tấn Nhất Tín,Lắp ráp trong nước,SUV,Xám,Đen,5.0,5.0,FWD - Dẫn động cầu trước,bonbanh.com,...,NaN,NaN,False,2024.0,45000.0,Used,VinFast,VF5 Plus,Hà Nội,VF5


In [89]:
df_final = df_final.rename(columns={'car_model_clean': 'base_model'})
df_final = df_final.rename(columns={'car_model': 'model_mode'})

In [90]:
logical_order = [
    'id',
    'brand',
    'base_model',
    'model_mode',  # e.g., VF8 Plus AWD (for detailed trim analysis)
    'year',

    'condition',
    'mileage_km',
    'battery_status',
    'is_accident_free',
    'has_aftermarket_mods',

    'body_type',
    'seats',
    'doors',
    'drivetrain',
    'origin',
    'exterior_color',
    'interior_color',

    'city',
    'seller_name',
    'post_date',
    'website',

    # Target
    'price_vnd',
]

In [91]:
df_final = df_final[logical_order]

In [94]:
df_final.head()

,id,brand,base_model,model_mode,year,condition,mileage_km,battery_status,is_accident_free,has_aftermarket_mods,...,doors,drivetrain,origin,exterior_color,interior_color,city,seller_name,post_date,website,price_vnd
0,0,VinFast,VF8,VF8 Plus AWD,2023.0,Used,50000.0,NaN,NaN,False,...,5.0,AWD - 4 bánh toàn thời gian,Lắp ráp trong nước,Đen,Đen,Hà Nội,GF Hoàng Quốc Việt,2026-02-28,bonbanh.com,659000000.0
1,1,VinFast,VF8,VF8 Eco,2026.0,New,0.0,NaN,NaN,False,...,5.0,FWD - Dẫn động cầu trước,Lắp ráp trong nước,Trắng,Đen,Hà Nội,Vinfast Times City,2026-02-28,bonbanh.com,790000000.0
2,2,VinFast,VF8,VF8 Plus AWD,2023.0,Used,99000.0,NaN,NaN,False,...,5.0,AWD - 4 bánh toàn thời gian,Lắp ráp trong nước,Xanh,Nhiều màu,Hà Nội,GF Hoàng Quốc Việt,2026-02-28,bonbanh.com,599000000.0
3,3,VinFast,VF8,VF8 Eco AWD,2023.0,Used,55000.0,NaN,NaN,False,...,5.0,AWD - 4 bánh toàn thời gian,Lắp ráp trong nước,Xám,Đen,Hà Nội,Đức Lộc,2026-02-28,bonbanh.com,580000000.0
4,4,VinFast,VF5,VF5 Plus,2024.0,Used,45000.0,NaN,NaN,False,...,5.0,FWD - Dẫn động cầu trước,Lắp ráp trong nước,Xám,Đen,Hà Nội,Tấn Nhất Tín,2026-02-28,bonbanh.com,399000000.0


In [95]:
new_car_mask = df_final['condition'] == 'New'

In [96]:
df_final.loc[new_car_mask, 'mileage_km'] = 0.0
df_final.loc[new_car_mask, 'is_accident_free'] = True
df_final.loc[new_car_mask, 'has_aftermarket_mods'] = False

In [101]:
df_final[new_car_mask][['condition', 'mileage_km', 'is_accident_free', 'has_aftermarket_mods']].head(5)

,condition,mileage_km,is_accident_free,has_aftermarket_mods
1,New,0.0,True,False
7,New,0.0,True,False
8,New,0.0,True,False
11,New,0.0,True,False
14,New,0.0,True,False


Fomr body_type, seats, doors, origin, drivetrain columns, we can see that they are available online. In this step, I filter them out and then fill them with information I can find online to have a more complete dataset before EDA.

In [102]:
static_cols = ['body_type', 'seats', 'doors', 'origin', 'drivetrain']

In [103]:
df_final[static_cols].isna().sum()

body_type      521
seats          271
doors         2388
origin         728
drivetrain    2387
dtype: int64

Phase A is about information would not change over years. We could use mode of each base_model to fill the missing values in these columns.

In [106]:
cols_phase_a = ['body_type', 'seats', 'doors', 'origin']

In [105]:
for col in cols_phase_a:
    df_final[col] = (df_final.groupby('base_model')[col]
                     .transform(lambda x: x.fillna(x.mode()[0]) if not x.mode().empty else x))

In [107]:
df_final[static_cols].isna().sum()

body_type       29
seats           25
doors           49
origin          17
drivetrain    2387
dtype: int64

Phase B focuses on the `drivetrain` column. Drivetrain information may vary according to year.

In [109]:
df_final['drivetrain'] = (df_final
                          .groupby('model_mode')['drivetrain']
                          .transform(lambda x: x.fillna(x.mode()[0]) if not x.mode().empty else x))

If a specific trim (model_mode) is so rare that it has no drivetrain data, we can fallback to using the mode of the base_model to fill in the missing drivetrain information. This way, we ensure that we are using the most relevant information available for each trim level while still maintaining consistency within the same base model.

In [110]:
df_final['drivetrain'] = (df_final
                          .groupby('base_model')['drivetrain']
                          .transform(lambda x: x.fillna(x.mode()[0]) if not x.mode().empty else x))

In [111]:
df_final['drivetrain'].isna().sum()

np.int64(49)

In [112]:
df_final[static_cols].isna().sum()

body_type     29
seats         25
doors         49
origin        17
drivetrain    49
dtype: int64

In [113]:
static_cols = ['body_type', 'seats', 'doors', 'origin', 'drivetrain']

In [127]:
missing_static_df = df_final[df_final[static_cols].isna().any(axis=1)]
missing_static_df['car_name'] = missing_static_df['brand'] + ' model ' + missing_static_df['base_model'] + ' mode ' + missing_static_df['model_mode']  + ' year ' + missing_static_df['year'].astype(str)

In [128]:
problem_models = missing_static_df['car_name'].unique()

In [129]:
problem_models

<StringArray>
[                                                                 nan,
                'Wuling model Mini 170Km mode Mini 170Km year 2023.0',
                    'Wuling model Ev 170 mode EV 170 LV2 year 2025.0',
                          'Toyota model Hiace mode Hiace year 2010.0',
                   'VinFast model E34 Plus mode e34 Plus year 2022.0',
                          'Hongqi model E-Hs9 mode E-HS9 year 2022.0',
                   'BYD model M9 Advance mode M9 Advance year 2025.0',
                     'BYD model Sealion 6 mode Sealion 6 year 2025.0',
                               'Geely model Ex5 mode Ex5 year 2025.0',
                   'VinFast model Limo 7 mode Limo 7 chỗ year 2025.0',
    'Wuling model Mini EV mode Hongguang Mini EV LV2-120 year 2023.0',
           'Volvo model Ec40 Recharge mode EC40 Recharge year 2025.0',
                               'Geely model Ex5 mode EX5 year 2025.0',
           'BYD model Atto3 2025 mode Atto3 2025 Dynamic year 2

In [130]:
master_specs_phase_c = {
    'Wuling model Mini 170Km mode Mini 170Km year 2023.0': {'body_type': 'Hatchback', 'seats': 4.0, 'doors': 3.0, 'origin': 'Lắp ráp trong nước', 'drivetrain': 'RWD - Dẫn động cầu sau'},
    'Wuling model Ev 170 mode EV 170 LV2 year 2025.0': {'body_type': 'Hatchback', 'seats': 4.0, 'doors': 3.0, 'origin': 'Lắp ráp trong nước', 'drivetrain': 'RWD - Dẫn động cầu sau'},
    'Toyota model Hiace mode Hiace year 2010.0': {'body_type': 'MPV', 'seats': 16.0, 'doors': 4.0, 'origin': 'Nhập khẩu', 'drivetrain': 'RWD - Dẫn động cầu sau'},
    'VinFast model E34 Plus mode e34 Plus year 2022.0': {'body_type': 'SUV', 'seats': 5.0, 'doors': 5.0, 'origin': 'Lắp ráp trong nước', 'drivetrain': 'FWD - Dẫn động cầu trước'},
    'Hongqi model E-Hs9 mode E-HS9 year 2022.0': {'body_type': 'SUV', 'seats': 7.0, 'doors': 5.0, 'origin': 'Nhập khẩu', 'drivetrain': 'AWD - 4 bánh toàn thời gian'},
    'BYD model M9 Advance mode M9 Advance year 2025.0': {'body_type': 'SUV', 'seats': 6.0, 'doors': 5.0, 'origin': 'Nhập khẩu', 'drivetrain': 'AWD - 4 bánh toàn thời gian'},
    'BYD model Sealion 6 mode Sealion 6 year 2025.0': {'body_type': 'SUV', 'seats': 5.0, 'doors': 5.0, 'origin': 'Nhập khẩu', 'drivetrain': 'FWD - Dẫn động cầu trước'},
    'Geely model Ex5 mode Ex5 year 2025.0': {'body_type': 'SUV', 'seats': 5.0, 'doors': 5.0, 'origin': 'Nhập khẩu', 'drivetrain': 'FWD - Dẫn động cầu trước'},
    'VinFast model Limo 7 mode Limo 7 chỗ year 2025.0': {'body_type': 'MPV', 'seats': 7.0, 'doors': 5.0, 'origin': 'Lắp ráp trong nước', 'drivetrain': 'FWD - Dẫn động cầu trước'},
    'Wuling model Mini EV mode Hongguang Mini EV LV2-120 year 2023.0': {'body_type': 'Hatchback', 'seats': 4.0, 'doors': 3.0, 'origin': 'Lắp ráp trong nước', 'drivetrain': 'RWD - Dẫn động cầu sau'},
    'Volvo model Ec40 Recharge mode EC40 Recharge year 2025.0': {'body_type': 'SUV', 'seats': 5.0, 'doors': 5.0, 'origin': 'Nhập khẩu', 'drivetrain': 'AWD - 4 bánh toàn thời gian'},
    'Geely model Ex5 mode EX5 year 2025.0': {'body_type': 'SUV', 'seats': 5.0, 'doors': 5.0, 'origin': 'Nhập khẩu', 'drivetrain': 'FWD - Dẫn động cầu trước'},
    'BYD model Atto3 2025 mode Atto3 2025 Dynamic year 2025.0': {'body_type': 'SUV', 'seats': 5.0, 'doors': 5.0, 'origin': 'Nhập khẩu', 'drivetrain': 'FWD - Dẫn động cầu trước'},
    'Geely model Coolray mode Coolray year 2025.0': {'body_type': 'SUV', 'seats': 5.0, 'doors': 5.0, 'origin': 'Nhập khẩu', 'drivetrain': 'FWD - Dẫn động cầu trước'},
    'Hyundai model Accent mode Accent year 2020.0': {'body_type': 'Sedan', 'seats': 5.0, 'doors': 4.0, 'origin': 'Lắp ráp trong nước', 'drivetrain': 'FWD - Dẫn động cầu trước'},
    'Volvo model Ec40 mode EC40 year 2025.0': {'body_type': 'SUV', 'seats': 5.0, 'doors': 5.0, 'origin': 'Nhập khẩu', 'drivetrain': 'AWD - 4 bánh toàn thời gian'},
    'Geely model Ex5 mode EX5 year 2024.0': {'body_type': 'SUV', 'seats': 5.0, 'doors': 5.0, 'origin': 'Nhập khẩu', 'drivetrain': 'FWD - Dẫn động cầu trước'},
    'VinFast model E34 mode E34 year 2022.0': {'body_type': 'SUV', 'seats': 5.0, 'doors': 5.0, 'origin': 'Lắp ráp trong nước', 'drivetrain': 'FWD - Dẫn động cầu trước'},
    'BYD model M9 Advanced mode M9 Advanced / Premium year 2025.0': {'body_type': 'SUV', 'seats': 6.0, 'doors': 5.0, 'origin': 'Nhập khẩu', 'drivetrain': 'AWD - 4 bánh toàn thời gian'},
    'BYD model Sealion 6 mode Sealion 6 Dynamic / Premium year 2025.0': {'body_type': 'SUV', 'seats': 5.0, 'doors': 5.0, 'origin': 'Nhập khẩu', 'drivetrain': 'FWD - Dẫn động cầu trước'},
    'BYD model M9 mode M9 year 2025.0': {'body_type': 'SUV', 'seats': 6.0, 'doors': 5.0, 'origin': 'Nhập khẩu', 'drivetrain': 'AWD - 4 bánh toàn thời gian'},
    'Jaecoo model J7 Awd mode J7 AWD Individual year 2025.0': {'body_type': 'SUV', 'seats': 5.0, 'doors': 5.0, 'origin': 'Nhập khẩu', 'drivetrain': 'AWD - 4 bánh toàn thời gian'},
    'BYD model Sealion 6 mode Sealion 6 Plug-in hybrid DM-i year 2025.0': {'body_type': 'SUV', 'seats': 5.0, 'doors': 5.0, 'origin': 'Nhập khẩu', 'drivetrain': 'FWD - Dẫn động cầu trước'},

    'VinFast model 900Lx mode 900LX year 2025.0': {'body_type': 'SUV', 'seats': 4.0, 'doors': 5.0, 'origin': 'Lắp ráp trong nước', 'drivetrain': 'AWD - 4 bánh toàn thời gian'},

    'Wuling model Mini EV mode HongGuang MiniEV year 2023.0': {'body_type': 'Hatchback', 'seats': 4.0, 'doors': 3.0, 'origin': 'Lắp ráp trong nước', 'drivetrain': 'RWD - Dẫn động cầu sau'},
    'Wuling model Mini Ev mode Mini EV LV2 year 2023.0': {'body_type': 'Hatchback', 'seats': 4.0, 'doors': 3.0, 'origin': 'Lắp ráp trong nước', 'drivetrain': 'RWD - Dẫn động cầu sau'},
    'Lexus model Lm500H mode LM500h year 2025.0': {'body_type': 'MPV', 'seats': 4.0, 'doors': 5.0, 'origin': 'Nhập khẩu', 'drivetrain': 'AWD - 4 bánh toàn thời gian'},
    'VinFast model Mvp 7 mode MVP 7 year 2026.0': {'body_type': 'MPV', 'seats': 7.0, 'doors': 5.0, 'origin': 'Lắp ráp trong nước', 'drivetrain': 'FWD - Dẫn động cầu trước'}
}

In [131]:
for idx, row in df_final.iterrows():
    model_mode_val = row.get('model_mode', row.get('car_model'))

    car_name_key = f"{row['brand']} model {row['base_model']} mode {model_mode_val} year {row['year']}"

    if car_name_key in master_specs_phase_c:
        for col, default_val in master_specs_phase_c[car_name_key].items():
            if pd.isna(row[col]):
                df_final.at[idx, col] = default_val

In [132]:
df_final[static_cols].isna().sum()

body_type     17
seats         17
doors         20
origin        17
drivetrain    20
dtype: int64

In [134]:
static_cols = ['body_type', 'seats', 'doors', 'origin', 'drivetrain']

# Find the exact rows still missing data
ghost_cars = df_final[df_final[static_cols].isna().any(axis=1)]

# Display their core info to see what's wrong with them
print(f"Total Ghost Cars: {len(ghost_cars)}")
display(ghost_cars[['id', 'brand', 'base_model', 'model_mode', 'year'] + static_cols])

Total Ghost Cars: 20


,id,brand,base_model,model_mode,year,body_type,seats,doors,origin,drivetrain
1622,1622,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1623,1623,NaN,NaN,NaN,2025.0,NaN,NaN,NaN,NaN,NaN
1809,1809,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1958,1958,BYD,Sealion 6,Sealion 6,NaN,SUV/Crossover,5.0,NaN,Trung Quốc,NaN
2060,2060,NaN,E34,E34,2022.0,SUV/Crossover,5.0,NaN,Việt Nam,NaN
2331,2331,VinFast,NaN,NaN,2025.0,NaN,NaN,NaN,NaN,NaN
2395,2395,VinFast,NaN,NaN,2025.0,NaN,NaN,NaN,NaN,NaN
2537,2537,VinFast,NaN,NaN,2026.0,NaN,NaN,NaN,NaN,NaN
2581,2581,NaN,NaN,NaN,2026.0,NaN,NaN,NaN,NaN,NaN
2865,2865,NaN,NaN,NaN,2025.0,NaN,NaN,NaN,NaN,NaN


In [135]:
df_final.loc[df_final['id'] == 1958, ['doors', 'drivetrain']] = [5.0, 'FWD - Dẫn động cầu trước']

In [136]:
df_final.loc[df_final['id'].isin([2060, 3081]),
             ['brand', 'base_model', 'car_model', 'doors', 'drivetrain', 'body_type']] = \
             ['VinFast', 'VF e34', 'VF e34', 5.0, 'FWD - Dẫn động cầu trước', 'SUV']

In [137]:
df_final = df_final.dropna(subset=['base_model'])

In [138]:
df_final[static_cols].isna().sum()

body_type     0
seats         0
doors         0
origin        0
drivetrain    0
dtype: int64

In [140]:
tolerance = 0.5
missing_pct = df_final.isnull().mean()
cols_to_drop = missing_pct[missing_pct > tolerance].index.tolist()

In [141]:
for col, pct in missing_pct.items():
    status = "[DROPPING]" if col in cols_to_drop else "[KEEPING]"
    print(f"{status} {col}: {pct:.2%}")

df_final = df_final.drop(columns=cols_to_drop)

[KEEPING] id: 0.00%
[KEEPING] brand: 0.68%
[KEEPING] base_model: 0.00%
[KEEPING] model_mode: 0.00%
[KEEPING] year: 1.69%
[KEEPING] condition: 11.63%
[KEEPING] mileage_km: 6.74%
[DROPPING] battery_status: 74.64%
[DROPPING] is_accident_free: 51.11%
[KEEPING] has_aftermarket_mods: 0.00%
[KEEPING] body_type: 0.00%
[KEEPING] seats: 0.00%
[KEEPING] doors: 0.00%
[KEEPING] drivetrain: 0.00%
[KEEPING] origin: 0.00%
[KEEPING] exterior_color: 4.83%
[DROPPING] interior_color: 59.64%
[KEEPING] city: 0.00%
[KEEPING] seller_name: 0.00%
[KEEPING] post_date: 0.00%
[KEEPING] website: 0.00%
[KEEPING] price_vnd: 0.05%
[DROPPING] car_model: 99.95%


In [142]:
print(f"Dropped {len(cols_to_drop)} columns. New shape: {df_final.shape}")

Dropped 4 columns. New shape: (3974, 19)


In [145]:
df_final["body_type"].value_counts()

body_type
SUV                 1412
SUV / Cross over     921
SUV/Crossover        732
Hatchback            560
Minivan (MPV)         98
Sedan                 82
Crossover             47
Hatback               21
Van                   21
Kiểu dáng khác        20
Coupe (2 cửa)         17
Van/Minivan           15
Mini                   8
Hatbach                8
SUV-B                  5
MPV                    3
Mini Car               3
Wagon                  1
Name: count, dtype: int64

In [146]:
body_type_map = {
    'SUV / Cross over': 'SUV',
    'SUV/Crossover': 'SUV',
    'Crossover': 'SUV',
    'SUV-B': 'SUV',

    'Hatback': 'Hatchback',
    'Hatbach': 'Hatchback',
    'Mini': 'Hatchback',
    'Mini Car': 'Hatchback',

    'Minivan (MPV)': 'MPV',
    'Van/Minivan': 'MPV',
    'Van': 'MPV',

    'Kiểu dáng khác': 'Other',
    'Coupe (2 cửa)': 'Coupe'
}

In [147]:
df_final['body_type'] = df_final['body_type'].replace(body_type_map)

In [148]:
df_final['body_type'].value_counts()

body_type
SUV          3117
Hatchback     600
MPV           137
Sedan          82
Other          20
Coupe          17
Wagon           1
Name: count, dtype: int64

In [149]:
OUTPUT_PATH = INTERIM / "ev_eda_ready.csv"

In [150]:
df_final.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')